# RAG Hands-on Lab


In this lab you will build a small **Retrieval Augmented Generation (RAG)** pipeline over an employee handbook, see where a basic RAG setup gives wrong answers, and fix it one step at a time.
Tasks:




1.   Setup
2.   Look at the document and the test questions
3. Run the **basic RAG** pipeline and score it
4. Each pair picks **one fix** and re-scores
5. Combine fixes, including the prompt, to reach 7/7




Each fix you try is a **module** in Modular RAG: chunking, retrieval size, hybrid search, query rewriting.

## Part 1. Setup

In [1]:
# Setup (installing chromadb can take about a minute)
!pip install -q openai rank_bm25 chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/13

### Add your OpenAI API key as a Colab secret (one time only)

1. Click the **key icon** (🔑, "Secrets") in the left sidebar.
2. Click **+ Add new secret**.
3. **Name:** `OPENAI_API_KEY`. **Value:** your OpenAI API key.
4. Turn on **Notebook access** for this notebook.

If you already added this secret for another course notebook, just turn on **Notebook access** here.

In [2]:
import os
import re
import json
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from rank_bm25 import BM25Okapi

def load_openai_key():
    """Read the OpenAI API key from Colab Secrets; ask for it only if that fails."""
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except ImportError:
        print("Not running in Colab.")
    except Exception as e:
        print(f"Could not read the Colab secret OPENAI_API_KEY ({type(e).__name__}).")
        print("Check: secret name, value, and Notebook access.")
    from getpass import getpass
    return getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = load_openai_key()
client = OpenAI()

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"
print("Ready.")

Ready.


## Part 2. The document and the test questions

Our knowledge source is a short (fictional) employee handbook. In real projects this would be many PDFs or web pages; the steps are the same.

In [3]:
HANDBOOK = """
Northwind Analytics Employee Handbook

1. Annual Leave
Full-time employees receive 18 days of annual leave per calendar year. Employees with more than five years of service receive 21 days.
Up to 5 unused days may be carried over to the next year. Carried-over days must be used by 31 March, after which they expire.
Leave must be requested in the HR portal at least 7 days in advance using form HR-17. Your manager approves or rejects the request within 3 working days.

2. Medical Leave
Employees are entitled to 14 days of outpatient medical leave and 60 days of hospitalisation leave per year, which includes the outpatient days.
A medical certificate from a registered doctor is required for every day of medical leave. Upload the certificate in the HR portal using form HR-21 within 2 working days of returning to work.

3. Working From Home
Employees may work from home up to 2 days per week with their manager's approval. Core hours are 10am to 4pm, when everyone must be reachable on chat and phone.
Working from home is not available during the first 3 months of employment.

4. Business Travel
All business travel must be approved by your manager in the travel system before any booking is made.
Economy class is used for flights under 6 hours. Premium economy may be booked for flights of 6 hours or more.
Hotel limits per night are: Singapore S$220, Tokyo S$260, London S$300, and S$200 for all other cities.
The overseas meal allowance is S$100 per day. For local trips within Singapore the meal allowance is S$30 per day.

5. IT Support
Passwords expire every 90 days. To reset a forgotten password, use the self-service portal at it.northwind.example and verify with your phone.
VPN error VPN-401 means your password has expired: reset your password, then reconnect.
VPN error VPN-403 means your device is not registered: submit form IT-22 to register the device. Registration takes up to 2 working days.
A lost or stolen laptop must be reported to the IT helpdesk within 24 hours.

6. Office Facilities
The office is open from 8am to 7pm on weekdays. After-hours access requires a security pass from Facilities.
Animals are not allowed in the office, except service animals.
Meeting rooms are booked through the calendar system and may be held for at most 3 hours.
"""
print(len(HANDBOOK.split()), "words")

390 words


### Test questions

Each question lists the facts a correct answer must contain (`|` separates accepted wordings). The last question is **not** covered by the handbook, so the right answer is "I don't know".

In [4]:
TESTS = [
    {"q": "How many unused leave days can I carry over, and by when must I use them?",
     "expect": ["5", "31 March|March 31"]},
    {"q": "What is form HR-21 used for?",
     "expect": ["medical"]},
    {"q": "My VPN shows VPN-403. What should I do?",
     "expect": ["IT-22"]},
    {"q": "Can I work from home three days a week?",
     "expect": ["2 days|two days"]},
    {"q": "What is the hotel limit per night in London?",
     "expect": ["300"]},
    {"q": "I forgot my login password. How do I get back in?",
     "expect": ["self-service|self service"]},
    {"q": "What is the company dress code?",
     "expect": ["don't know|do not know"]},
]

## Part 3. The RAG pipeline

The pipeline has four steps. Read each function, then run the cell.

**Indexing (done once):** split the document into chunks → add them to a **vector database** (Chroma), which embeds and stores them.
**Answering (per question):** find the most relevant chunks → give them to the LLM with the question.

### Step 1: Chunking

In [5]:
def chunk_text(text, chunk_size=40, overlap=0):
    """Split text into chunks of `chunk_size` words; neighbouring chunks share `overlap` words."""
    words = text.split()
    step = max(chunk_size - overlap, 1)
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
    return chunks

sample = chunk_text(HANDBOOK, chunk_size=40, overlap=0)
print(len(sample), "chunks. First two:\n")
for c in sample[:2]:
    print("-", c, "\n")

10 chunks. First two:

- Northwind Analytics Employee Handbook 1. Annual Leave Full-time employees receive 18 days of annual leave per calendar year. Employees with more than five years of service receive 21 days. Up to 5 unused days may be carried over to the 

- next year. Carried-over days must be used by 31 March, after which they expire. Leave must be requested in the HR portal at least 7 days in advance using form HR-17. Your manager approves or rejects the request within 3 



### Step 2: Store the chunks in a vector database

We use **Chroma**, an open-source vector database, in the way it is typically used for RAG:

- A **collection** holds one searchable set of documents (here: the handbook).
- The collection has an **embedding function**, so Chroma embeds the chunks when we add them and embeds the question when we search. Documents and questions are guaranteed to use the same model.
- Each chunk is stored with an **id**, its **text**, its **embedding**, and **metadata** (here: chunk number and handbook section).

We set the embedding function to the same OpenAI model as in the embeddings demo. If you don't set one, Chroma silently uses its own small default model, and results would differ.

> In the lab, Chroma runs **in memory** (`chromadb.Client()`). A real app would use `chromadb.PersistentClient(path="./chroma_db")` to keep the index on disk, or a Chroma server shared by several apps.

In [6]:
# Chroma calls OpenAI for us, reading the key from the OPENAI_API_KEY environment variable
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    model_name=EMBED_MODEL,
)

SECTIONS = ["Annual Leave", "Medical Leave", "Working From Home", "Business Travel", "IT Support", "Office Facilities"]

def section_of(chunks):
    """The handbook section each chunk mostly belongs to (stored as metadata)."""
    current, result = "Introduction", []
    for chunk in chunks:
        words = len(chunk.split())
        found = sorted((chunk.index(f"{n}. {name}"), name)
                       for n, name in enumerate(SECTIONS, 1) if f"{n}. {name}" in chunk)
        for pos, name in found:
            if len(chunk[:pos].split()) < words / 2:   # heading in the first half: chunk belongs to it
                current = name
        result.append(current)
        if found:
            current = found[-1][1]                      # the next chunk continues the last section seen
    return result

STOPWORDS = {"a", "an", "the", "is", "are", "be", "to", "of", "in", "on", "for", "and", "or", "with",
             "what", "how", "do", "does", "i", "my", "me", "can", "it", "this", "that", "used", "by", "at"}

def tokenize(text):
    """Lower-case words for keyword search, keeping codes like HR-21 together and dropping common words."""
    return [w for w in re.findall(r"[\w-]+", text.lower()) if w not in STOPWORDS]

db = chromadb.Client()      # in-memory Chroma database
_index_cache = {}

def build_index(chunk_size, overlap):
    """Chunk the handbook and add the chunks to a Chroma collection (Chroma embeds them).
    One collection per chunking setting, cached so other settings don't re-embed."""
    key = (chunk_size, overlap)
    if key not in _index_cache:
        chunks = chunk_text(HANDBOOK, chunk_size, overlap)
        name = f"handbook_{chunk_size}_{overlap}"
        try:
            db.delete_collection(name)          # start clean if the cell is re-run
        except Exception:
            pass
        collection = db.create_collection(name=name, embedding_function=openai_ef)
        collection.add(                          # Chroma embeds the documents for us
            ids=[f"chunk-{i}" for i in range(len(chunks))],
            documents=chunks,
            metadatas=[{"chunk": i, "section": sec} for i, sec in enumerate(section_of(chunks))],
        )
        bm25 = BM25Okapi([tokenize(c) for c in chunks])   # keyword index (outside the vector DB)
        _index_cache[key] = {"chunks": chunks, "collection": collection, "bm25": bm25}
        print(f"Stored {collection.count()} chunks in Chroma collection '{name}'")
    return _index_cache[key]

#### Look inside the vector database

Build the naïve index once and inspect what Chroma stores for one chunk.

In [7]:
index = build_index(chunk_size=40, overlap=0)
collection = index["collection"]

item = collection.get(ids=["chunk-0"], include=["documents", "metadatas", "embeddings"])
print("Text:     ", item["documents"][0][:100], "...")
print("Metadata: ", item["metadatas"][0])
print("Embedding:", len(item["embeddings"][0]), "numbers, first 5 =", np.round(item["embeddings"][0][:5], 4))

Stored 10 chunks in Chroma collection 'handbook_40_0'
Text:      Northwind Analytics Employee Handbook 1. Annual Leave Full-time employees receive 18 days of annual  ...
Metadata:  {'section': 'Annual Leave', 'chunk': 0}
Embedding: 1536 numbers, first 5 = [-0.0162  0.0644  0.0826  0.0209 -0.0166]


In [8]:
# Ask the vector database directly: the 3 chunks closest in meaning to a question.
# We pass the question as TEXT; Chroma embeds it with the same model as the chunks.
results = collection.query(
    query_texts=["How do I reset my password?"],
    n_results=3,
)
for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"distance {dist:.3f} | {meta['section']:<18} | {doc[:70]}...")
# Smaller distance = closer in meaning

distance 0.625 | IT Support         | use the self-service portal at it.northwind.example and verify with yo...
distance 0.670 | Business Travel    | S$300, and S$200 for all other cities. The overseas meal allowance is ...
distance 0.746 | IT Support         | Registration takes up to 2 working days. A lost or stolen laptop must ...


In [9]:
# Vector databases can also FILTER by metadata before searching.
# "How many days?" is vague: without a filter it could match leave, travel or IT chunks.
print("Without filter:")
results = collection.query(query_texts=["How many days?"], n_results=1)
print("  ", results["metadatas"][0][0]["section"], "|", results["documents"][0][0][:70], "...")

print("With filter section = 'Medical Leave':")
results = collection.query(query_texts=["How many days?"], n_results=1,
                           where={"section": "Medical Leave"})
print("  ", results["metadatas"][0][0]["section"], "|", results["documents"][0][0][:70], "...")

Without filter:
   Medical Leave | working days. 2. Medical Leave Employees are entitled to 14 days of ou ...
With filter section = 'Medical Leave':
   Medical Leave | working days. 2. Medical Leave Employees are entitled to 14 days of ou ...


In [10]:
# Chroma can also filter on the TEXT itself: only chunks that contain an exact string.
# Useful for codes like HR-21, which vector search alone may miss.
results = collection.query(
    query_texts=["What is this form for?"],
    n_results=1,
    where_document={"$contains": "HR-21"},
)
print(results["metadatas"][0][0]["section"], "|", results["documents"][0][0][:90], "...")

Medical Leave | of medical leave. Upload the certificate in the HR portal using form HR-21 within 2 workin ...


### Step 3: Retrieval

- **vector**: ask Chroma for the chunks closest in meaning to the question.
- **hybrid**: also rank by exact words (BM25 keyword search), then merge both rankings with *reciprocal rank fusion*. Chroma ranks by meaning and can *filter* on exact text (`where_document`, shown above), but it doesn't *rank* by keyword relevance, so the BM25 keyword search runs alongside it. In practice, hybrid search often combines two components like this.
- **rewrite**: ask the LLM for other ways to phrase the question, search with each, and merge the results.

In [11]:
def vector_ranking(index, query):
    """All chunk numbers, closest in meaning first (from Chroma)."""
    collection = index["collection"]
    results = collection.query(
        query_texts=[query],
        n_results=collection.count(),
        include=["metadatas"],
    )
    return [m["chunk"] for m in results["metadatas"][0]]

def keyword_ranking(index, query):
    scores = index["bm25"].get_scores(tokenize(query))
    return list(np.argsort(-scores))

def fuse(rankings, k=60):
    """Reciprocal rank fusion: a chunk ranked high in any list gets a high combined score."""
    combined = {}
    for ranking in rankings:
        for rank, i in enumerate(ranking):
            combined[i] = combined.get(i, 0) + 1 / (k + rank + 1)
    return sorted(combined, key=combined.get, reverse=True)

def rewrite_query(question, n=3):
    """Ask the LLM for n alternative phrasings of the question."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[{"role": "user", "content":
            f"Write {n} different short search queries that would help answer this question. "
            f"Return JSON: {{\"queries\": [...]}}\n\nQuestion: {question}"}],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)["queries"][:n]

def retrieve(question, config):
    index = build_index(config["chunk_size"], config["overlap"])
    queries = [question] + (rewrite_query(question) if config["rewrite"] else [])
    rankings = []
    for q in queries:
        rankings.append(vector_ranking(index, q))
        if config["search"] == "hybrid":
            rankings.append(keyword_ranking(index, q))
    top = fuse(rankings)[: config["top_k"]]
    return [(int(i), index["chunks"][i]) for i in top]

### Step 4: Generate the answer from the retrieved chunks

In [14]:
SYSTEM_PROMPT = """You answer employee questions using ONLY the handbook extracts provided.
If the extracts do not contain the answer, reply exactly: I don't know.
Keep the answer short and cite the extract numbers you used, like [2]."""

# A second prompt, used in Part 5
APPLY_RULES_PROMPT = """You answer employee questions using ONLY the handbook extracts provided.
If the extracts contain a rule that answers the question, apply it and explain briefly
(for example, a limit of 2 means 3 is not allowed).
Only if the extracts contain nothing relevant to the question, reply exactly: I don't know.
Keep the answer short and cite the extract numbers you used, like [2]."""

PROMPTS = {"strict": SYSTEM_PROMPT, "apply_rules": APPLY_RULES_PROMPT}

def answer(question, chunks, system_prompt=SYSTEM_PROMPT):
    context = "\n\n".join(f"[{i}] {text}" for i, text in chunks)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Handbook extracts:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content

def rag(question, config, show_chunks=False):
    chunks = retrieve(question, config)
    if show_chunks:
        for i, text in chunks:
            print(f"   chunk [{i}]: {text[:120]}...")
    prompt = PROMPTS[config.get("prompt", "strict")]   # strict unless the config says otherwise
    return answer(question, chunks, prompt)

### Score the pipeline on all test questions

In [15]:
def evaluate(config, show_chunks=False):
    print("Config:", config, "\n")
    passed = 0
    for t in TESTS:
        reply = rag(t["q"], config, show_chunks)
        text = reply.lower().replace("’", "'")
        ok = all(any(alt.lower() in text for alt in e.split("|")) for e in t["expect"])
        passed += ok
        print(("✅" if ok else "❌"), t["q"])
        print("   ->", reply.replace("\n", " "), "\n")
    print(f"Score: {passed}/{len(TESTS)}")
    return passed

### Run the basic RAG baseline

Small chunks, no overlap, only the single best chunk, vector search only. Run it and note the score and which questions fail.

In [16]:
BASIC_PIPELINE= {"chunk_size": 40, "overlap": 0, "top_k": 1, "search": "vector", "rewrite": False}

baseline = evaluate(BASIC_PIPELINE, show_chunks=True)

Config: {'chunk_size': 40, 'overlap': 0, 'top_k': 1, 'search': 'vector', 'rewrite': False} 

   chunk [1]: next year. Carried-over days must be used by 31 March, after which they expire. Leave must be requested in the HR portal...
❌ How many unused leave days can I carry over, and by when must I use them?
   -> You can carry over unused leave days, but they must be used by 31 March, after which they expire [1]. 

   chunk [3]: of medical leave. Upload the certificate in the HR portal using form HR-21 within 2 working days of returning to work. 3...
✅ What is form HR-21 used for?
   -> Form HR-21 is used to upload the medical certificate in the HR portal within 2 working days of returning to work after medical leave [3]. 

   chunk [7]: use the self-service portal at it.northwind.example and verify with your phone. VPN error VPN-401 means your password ha...
✅ My VPN shows VPN-403. What should I do?
   -> You should submit form IT-22 to register your device [7]. 

   chunk [4]: Core hou

**Look at the ❌ questions.** For each one, check the retrieved chunk:
- Was the needed fact **cut in half** by chunking?
- Was the right chunk **not retrieved at all**?
- Did the question need **two different chunks**?

> Results can vary a little between runs and models. What matters is *why* each failure happens.

## Part 4. Fix it: each pair picks ONE fix

Copy `BASIC_PIPELIE`, change **one** setting, and re-run `evaluate`. Write down the new score and which questions it fixed.

| Fix | Change | Modular RAG module | Typical effect |
|---|---|---|---|
| **A. Chunking** | `chunk_size` 120, `overlap` 30 | Indexing | Facts stay together in one chunk |
| **B. Top-k** | `top_k` 3 | Retrieval | Questions needing two chunks get both |
| **C. Hybrid search** | `search` "hybrid" | Retrieval (keyword + vector) | Exact codes like *HR-21* are found |
| **D. Query rewrite** | `rewrite` True | Pre-retrieval | Differently-worded questions still match |

In [17]:
# TODO Fix A: bigger chunks with overlap
fix_a = dict(BASIC_PIPELINE, chunk_size="TODO", overlap="TODO")
evaluate(fix_a, show_chunks=True)

Config: {'chunk_size': '', 'overlap': '', 'top_k': 1, 'search': 'vector', 'rewrite': False} 



TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [ ]:
# TODO Fix B: retrieve more chunks
fix_b = dict(BASIC_PIPELINE, top_k="TODO")
evaluate(fix_b, show_chunks=True)

Config: {'chunk_size': 40, 'overlap': 0, 'top_k': 3, 'search': 'vector', 'rewrite': False} 

   chunk [1]: next year. Carried-over days must be used by 31 March, after which they expire. Leave must be requested in the HR portal...
   chunk [2]: working days. 2. Medical Leave Employees are entitled to 14 days of outpatient medical leave and 60 days of hospitalisat...
   chunk [0]: Northwind Analytics Employee Handbook 1. Annual Leave Full-time employees receive 18 days of annual leave per calendar y...
✅ How many unused leave days can I carry over, and by when must I use them?
   -> You can carry over up to 5 unused leave days, and they must be used by 31 March, after which they expire [0][1]. 

   chunk [3]: of medical leave. Upload the certificate in the HR portal using form HR-21 within 2 working days of returning to work. 3...
   chunk [1]: next year. Carried-over days must be used by 31 March, after which they expire. Leave must be requested in the HR portal...
   chunk [0]: Northw

5

In [ ]:
# TODO Fix C: hybrid search (keyword + vector)
fix_c = dict(BASIC_PIPELINE, search="TODO")
evaluate(fix_c, show_chunks=True)

Config: {'chunk_size': 40, 'overlap': 0, 'top_k': 1, 'search': 'hybrid', 'rewrite': False} 

   chunk [1]: next year. Carried-over days must be used by 31 March, after which they expire. Leave must be requested in the HR portal...
❌ How many unused leave days can I carry over, and by when must I use them?
   -> You can carry over unused leave days, but they must be used by 31 March, after which they expire [1]. 

   chunk [3]: of medical leave. Upload the certificate in the HR portal using form HR-21 within 2 working days of returning to work. 3...
✅ What is form HR-21 used for?
   -> Form HR-21 is used to upload the medical certificate within 2 working days of returning to work after medical leave [3]. 

   chunk [7]: use the self-service portal at it.northwind.example and verify with your phone. VPN error VPN-401 means your password ha...
✅ My VPN shows VPN-403. What should I do?
   -> You should submit form IT-22 to register your device [7]. 

   chunk [4]: Core hours are 10am to 4p

4

In [ ]:
# TODO Fix D: query rewriting
fix_d = dict(BASIC_PIPELINE, rewrite="TODO")
evaluate(fix_d, show_chunks=True)

Config: {'chunk_size': 40, 'overlap': 0, 'top_k': 1, 'search': 'vector', 'rewrite': True} 

   chunk [1]: next year. Carried-over days must be used by 31 March, after which they expire. Leave must be requested in the HR portal...
❌ How many unused leave days can I carry over, and by when must I use them?
   -> You can carry over unused leave days, but they must be used by 31 March, after which they expire [1]. 

   chunk [3]: of medical leave. Upload the certificate in the HR portal using form HR-21 within 2 working days of returning to work. 3...
✅ What is form HR-21 used for?
   -> Form HR-21 is used to upload the medical certificate within 2 working days of returning to work [3]. 

   chunk [7]: use the self-service portal at it.northwind.example and verify with your phone. VPN error VPN-401 means your password ha...
✅ My VPN shows VPN-403. What should I do?
   -> You should submit form IT-22 to register your device [7]. 

   chunk [4]: Core hours are 10am to 4pm, when everyone must

4

## Part 5. Combine the fixes


Bigger chunks with overlap (A), more chunks (B) and hybrid search (C) together. Run it: does it reach 7/7?

In [ ]:
# TODO Step 1: combine the fixes that helped
best = {"chunk_size":"TODO", "overlap": "TODO", "top_k": "TODO", "search": "TODO", "rewrite": "TODO"}
evaluate(best)

Config: {'chunk_size': 120, 'overlap': 30, 'top_k': 3, 'search': 'hybrid', 'rewrite': False} 

✅ How many unused leave days can I carry over, and by when must I use them?
   -> You can carry over up to 5 unused leave days, and they must be used by 31 March after which they expire [0]. 

✅ What is form HR-21 used for?
   -> Form HR-21 is used to upload a medical certificate in the HR portal for every day of medical leave taken [1]. 

✅ My VPN shows VPN-403. What should I do?
   -> You need to submit form IT-22 to register your device. Registration takes up to 2 working days [3]. 

✅ Can I work from home three days a week?
   -> No, you may work from home up to 2 days per week with your manager's approval [1]. 

✅ What is the hotel limit per night in London?
   -> The hotel limit per night in London is S$300 [2]. 

✅ I forgot my login password. How do I get back in?
   -> To reset a forgotten password, use the self-service portal at it.northwind.example and verify with your phone [3]. 



7

## Part 6 (Optional)

You will probably get **6/7**. Look at the failing question: usually *"Can I work from home three days a week?"*, answered with *"I don't know."*

**Was it a retrieval problem?** Print the full chunks that were retrieved for that question:

In [ ]:
question = "Can I work from home three days a week?"
for i, text in retrieve(question, best):
    print(f"[{i}] {text}\n")

One of the chunks contains *"Employees may work from home up to **2 days per week**"*. So **retrieval worked**: the model had the rule.

The problem is in **generation**. Our prompt says *"If the extracts do not contain the answer, reply exactly: I don't know."* The handbook never literally says "three days is not allowed", so a strict model refuses, even though the answer follows from the rule.

No retrieval fix can solve this. We need to change a different module: the **prompt** used to generate the answer.

### Fix the generation step

`APPLY_RULES_PROMPT` (defined in Step 4) lets the model apply a rule it finds, and says "I don't know" only when nothing relevant was retrieved:

In [ ]:
print(APPLY_RULES_PROMPT)

In [ ]:
# Same retrieval settings, plus the improved prompt
best_full = dict(best, prompt="apply_rules")
evaluate(best_full)

This should reach **7/7**, and the dress code question still answers "I don't know", because nothing relevant is retrieved for it.

> Results can vary a little between runs and models. If a question still fails, print its chunks as above and decide: was it **retrieval** (fact not found) or **generation** (fact found, answered badly)?

### Discussion

1. Which single fix helped the most? Why?
2. Fix B (more chunks) helps recall, but what is the cost? *(Hint: tokens, cost, and noise in the context.)*
3. In Part 5, why couldn't any retrieval fix solve the work-from-home question? What does that tell you about where RAG can fail?
4. The strict prompt refused a question it could answer; a loose prompt might invent answers. Is it better for a RAG system to **refuse too often** or to **guess too often**? Who decides?
5. The dress code question should always return "I don't know". Which part of the pipeline makes that happen: retrieval or the prompt?
6. Map each fix to the **Modular RAG** slide: which module did you add or change?



### Take-away

Basic RAG is one fixed path. Each fix adds or tunes a **module**: chunking, retrieval size, hybrid search, query rewriting, and the **generation prompt**. A RAG answer can fail in **retrieval** (the fact isn't found) or in **generation** (the fact is found but used badly), and each needs a different fix. Choosing and combining modules for your data is **Modular RAG**, and measuring with a test set (like `TESTS`) is how you know a change really helped.